<div style="border-left:4px solid #34d399;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#34d399;">Understanding</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">How an English question becomes a schema, a symbol and an exact value.</div></div>

<div style="font:400 15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#3f3f46;">Everything on this page happens on this machine. Nothing has left yet, and by the end the question is ready to be sent with every real value removed.</div>

In [ ]:
# The code comes from GitHub. The repository is private, so this needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
import os, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")
if not ROOT.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT)

In [ ]:
# Notebook 1 saved the database, the index and the model weights. Reuse them
# instead of rebuilding: this cell is why notebooks 2-5 start in seconds.
import shutil
from pathlib import Path

SETUP = Path("/kaggle/input/nl2sql-1-setup")
if not SETUP.exists():
    raise SystemExit("Run notebook 1 (Setup) first, then add it as an input to this one.")

for name in ("data", "models"):
    source, target = SETUP / name, ROOT / name
    if source.exists() and not target.exists():
        shutil.copytree(source, target)

print("database:", (ROOT / "data" / "eicu.db").exists())
print("index:   ", (ROOT / "data" / "index.db").exists())

In [ ]:
# Keys live in Kaggle secrets, never in the notebook.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "LANGSMITH_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"{name} not set - the steps that need it will say so")

os.environ["LANGSMITH_TRACING"] = "1"
os.environ["LANGSMITH_PROJECT"] = "nl2sql"

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">1.</span> The database</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">What there is to ask about.</div></div>

In [ ]:
from nl2sql.db import schema

summary = schema.summary()
print(f"{summary['tables']} tables, {summary['columns']} columns, "
      f"{summary['rows']:,} rows, {summary['foreign_keys']} declared relationships")

print(schema.ddl({"patient", "medication"}))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">2.</span> The index</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Two tiers. A column is indexed when its vocabulary is bounded, resolved on demand when it is not.</div></div>

In [ ]:
from nl2sql.db import values

stats = values.stats()
print("tier A (indexed) :", stats["tiers"].get("A"), "columns")
print("tier B (on demand):", stats["tiers"].get("B"), "columns")
print("values indexed   :", f"{stats['values_indexed']:,}", f"({stats['size_mb']} MB)")
print()
for ref, n in stats["top"][:6]:
    print(f"  {ref:<44} {n:>6}")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">The cost of the index is bounded by the number of columns, not by the number of rows. That is what makes it transfer to a database far larger than this one.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">3.</span> Finding a value</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The analyst writes a word; the database holds something longer.</div></div>

In [ ]:
for mention in ["aspirin", "asspirin", "sepsis", "female"]:
    found = values.search(mention, limit=1)
    if found:
        best = found[0]
        print(f"{mention:<12} -> {best.value[:46]:<48} {best.ref:<28} {best.score:.2f}")
    else:
        print(f"{mention:<12} -> nothing")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">4.</span> Naming a column</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Not every word is content. Some name a column, and the two compete.</div></div>

In [ ]:
from nl2sql.db import catalog

for mention in ["diagnosis names", "administration routes", "aspirin"]:
    match = catalog.best(mention)
    if match:
        print(f"{mention:<24} -> {match.ref:<34} {match.score:.2f}  ({match.why})")
    else:
        print(f"{mention:<24} -> no column matches it, so it can only be a value")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">A real value scores low against every column, and a column name scores low against every value. That gap is what lets the pipeline tell them apart instead of guessing.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">5.</span> Reading the question</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The three previous steps, run together, with every decision recorded.</div></div>

In [ ]:
from nl2sql.core import trace
from nl2sql.nlp.understand import understand

trace.configure()
with trace.record('How many patients over 65 received aspirin?') as run:
    u = understand('How many patients over 65 received aspirin?')

for step in run.steps:
    print(f"  {step.label:<40} {step.ms:>7.0f} ms  {step.summary}")

In [ ]:
print("tables:", sorted(u.tables))
print()
for r in u.resolutions:
    kind = r.kind.upper()
    value = f"-> {r.value}" if r.value else ""
    print(f"  {r.mention:<16} {kind:<10} {str(r.column):<34} {value}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">6.</span> Hiding the values</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Each value becomes a symbol. The mapping stays here.</div></div>

In [ ]:
from nl2sql.privacy.mask import mask

masked = mask(u)
print("before:", u.question)
print("after :", masked.question)
print()
for symbol, value in masked.mapping.items():
    print(f"  {symbol} = {value!r}   from {masked.columns.get(symbol, 'the analyst')}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">7.</span> The message that would be sent</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Assembled, then checked part by part before any connection is opened.</div></div>

In [ ]:
from nl2sql.core import prompt
from nl2sql.privacy import gate

built = prompt.hybrid(u, masked)
for verdict in gate.verdicts(built.segments):
    mark = "ok " if verdict["allowed"] else "NO "
    print(f"  {mark}{verdict['origin']:<10} {verdict['checked_by']:<30} {verdict['preview'][:52]}")

In [ ]:
print(built.messages[-1]["content"])

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">Not one real value appears above. The provider is given the shape of the database and a sentence with holes in it.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">8.</span> What it refuses</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Two questions that cannot be answered, and are stopped here rather than half-answered.</div></div>

In [ ]:
from nl2sql.privacy.mask import UnmaskableQuestion, UnresolvableValue

for question in ["Did Mr. Bensalah receive insulin?", "How many patients received asparatan?"]:
    try:
        mask(understand(question))
        print(f"{question}\n  -> sent\n")
    except (UnmaskableQuestion, UnresolvableValue) as e:
        print(f"{question}\n  -> {e}\n")